# 26 · 读病历：把医生写的字变成 agent 能用的 HPO 编号

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → [26 读病历 ✅]
```

> **这一章的代码在合并请求 #27 里**（分支 `feat/note-to-hpo`）。花费：开发约 $0.15，正式考试 $0.37，演示约 $0.01。**运行这本笔记基本不花钱**：结果都从 MLflow 和保存的文件里读；只有第 2 步有一格可选的"现场读一份病历"，约 ¥0.006，不想花可以跳过。

## 背景：医生不写编号

到第 25 章为止，agent 吃的都是 **HPO 编号**：`HP:0001252`、`HP:0001250`……评测用的病人（Phenopacket Store）本来就是编码好的，所以一直没出问题。

可是真实的医生写的是这样的字：

> *In the newborn period she had prolonged jaundice, with vomiting and feeding difficulties. She was floppy as an infant. … Hearing screening was normal, and there are no cataracts. … A maternal cousin has epilepsy.*

没有一个编号。要用 agent，得有人先把这段话翻成编号——遗传科医生手动查，一份病历十几分钟。**这一步不解决，agent 在临床上就用不起来。** DeepRare 这类系统也都把"读病历"做成了第一步。

这一步看着简单，其实有三个坑，上面那段话里全有：

| 坑 | 例子 | 读错了会怎样 |
|---|---|---|
| **换个说法** | "floppy"（软）= 肌张力低下 Hypotonia | 字典里查不到，漏掉 |
| **否认** | "there are **no** cataracts" | 当成"有白内障"送进去，诊断被带偏 |
| **不是病人自己的** | "a maternal **cousin** has epilepsy" | 当成病人有癫痫，同样带偏 |

再加一个大模型特有的坑：**让模型直接写编号，它会编。** HPO 有一万九千个表型词条，编号是七位数字，模型记不住，会写出一个看起来很像、其实是别的词条（甚至不存在）的编号。

**公司里的做法**：这一步在业界叫**临床实体识别和标准化**（clinical NER + entity linking / normalization）。现在的主流做法是"**大模型负责理解，本体负责编号**"：模型读懂文字，候选编号从本体里检索出来，模型只能在候选里挑。行话叫 **grounding**（让输出落在真实存在的东西上）。

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 找一把公开的尺子 | 人工标注的公开语料 ID-68、GSC-2024；下载登记、校验 | 自己出题自己考不算数；能和已发表的方法比 |
| 2 | 三种读法 | 字典；模型直接写编号；我们的：模型读、本体编码、字典核对 | 每一个对照回答一个问题 |
| 3 | HPO 词条搜索 | 名字和同义词，混合检索（复用第 18 章的 Qdrant + FastEmbed） | 给模型候选，让它不用自己写编号 |
| 4 | 在开发集上改三版 | 看错题；只改"该改的" | 提高分数，又不去迎合标注者的习惯 |
| 5 | 先写方案，再考 | 预注册；考前发现的改动写进补充 | 结果可信 |
| 6 | 考试结果 | 文档级 F1，逐份配对检验 | 回答方案里的问题 |
| 7 | 做进服务，病历不留底 | MLflow 的 span processor（记录前抹掉）；默认关闭 | 病历是最敏感的数据 |
| 8 | 演示：贴一段病历 → 出诊断 | Streamlit 页面：标出原话、勾选、诊断 | 录屏用的完整流程 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 找一把公开的尺子

### 在干什么
找别人已经人工标注好的病历：每一份都有专家标的"这段话里有哪些 HPO 词条"。拿我们的读法去读，和专家的答案对。

### 背景
自己写几份病历、自己标、自己考，分数多少都没人信：病历可能写得刚好适合自己的读法。所以和前几章一样，**尺子要用别人的**。而且用别人的尺子，就能和别人发表的方法直接比。

### 用了什么技术，为什么
两份公开语料，都是这个领域论文里的标准考题：

| 语料 | 内容 | 谁标的 | 我们拿来做什么 |
|---|---|---|---|
| **ID-68** | 68 份**真实临床病历**（智力障碍家系，Anazi 等 2017 年发表） | PhenoBERT 的作者 | **主要考试**：最像"医生贴进来的病历" |
| **GSC-2024** | 228 篇 PubMed 摘要（罕见病论文） | 2017 年标注，2024 年按新版 HPO 重标 | 次要考试：206 篇考，22 篇留作开发 |
| GeneReviews | 10 篇疾病综述摘要 | 和 ID-68 同一批人 | 开发用 |

下载方式和第 05 章所有数据一样：在 `evals/datasets.toml` 里登记版本和 sha256 校验码，`raredx data fetch` 下载并校验，对不上就删掉。**直接从压缩包里读，不解压、不运行里面的任何东西**——下载来的东西都当作不可信的数据。

### 公司里怎么做
评测数据要固定版本、能追溯来源、有校验码，叫 **data provenance**。用公开基准（public benchmark）和已发表的方法比，是写技术报告、过评审的基本要求。

### 执行
先看看一份 ID-68 病历长什么样，和专家标了什么。

In [ ]:
from pathlib import Path
from raredx.phenotyping import corpora
A = Path("data/raw/phenobert/195d5df/195d5df2ad9c6265e1de523f139694c7646eee29.zip")
T = Path("data/raw/gsc-2024/77b5761/gsc_2024.tar.gz")
for c in corpora.CORPORA:
    docs = corpora.load(c, A, T)
    print(f"{c:<14} {len(docs):>4} 份   人工标注 {sum(len(d.gold) for d in docs):>5} 处")
note = corpora.load("id-68", A, T)[0]
print("\n── ID-68 的第一份病历 ──\n")
print(note.text[:900], "…")
print("\n── 专家标的（前 12 处）──")
for a in note.gold[:12]:
    print(f"  {a.hpo_id}  {a.text!r:<32} {'（否认）' if a.kind == 'negated' else ''}")

### 结果怎么看
- 这份病历里，"barely can feel pain"（几乎感觉不到疼）、"no tears"（哭的时候没有眼泪）被标为**否认**——这正是上面说的第二个坑
- "she was noted to be floppy with poor sucking"：专家只标了 poor sucking（吸吮无力），"floppy"（软，肌张力低下）没有标。**专家标注也不完美**——读法找到了 floppy，在这份病历上反而会被算"多出来"。所有读法都受同样的影响，这一点第 4 步还会碰到

---

## 第 2 步 · 三种读法

### 在干什么
写三种把病历读成编号的办法，每一种回答一个问题。

### 背景
只拿出一种读法说"F1 是多少"，别人会问："那不用大模型行不行？""那直接让大模型写编号行不行？"要回答，就得把这两种也做出来，放在同一把尺子上量。

### 用了什么技术，为什么

| 读法 | 怎么做 | 回答什么问题 |
|---|---|---|
| **字典**（`dictionary`） | 病历里出现的 HPO 名字和同义词，最长的优先；不用缩写 | 不用大模型，能到多少？ |
| **模型直接写编号**（`model-direct`） | 一次调用："列出表型和它们的 HPO 编号" | 大模型自己就够了吗？ |
| **我们的**（`note-reader`） | 下面四步 | 两者结合，好多少？ |

我们的读法，拿第 8 步演示用的那份病历举例：

1. **模型读**：列出每个表型，附上**病历原话**、用临床术语怎么说、病人是有／没有／亲属有。比如 原话 "floppy" → 术语 "Hypotonia" → 有；原话 "no cataracts" → "Cataract" → **没有**；原话 "maternal cousin has epilepsy" → "Seizure" → **亲属**
2. **名字对得上的直接编码**："Vomiting" 正好是 HPO 的名字，直接用 `HP:0002013`，不再问模型
3. **对不上的，检索候选**："Hypergalactosemia" 这类，在 HPO 的名字和同义词里检索出十来个候选（第 3 步），**模型只能从候选里挑**，不能自己写编号
4. **字典核对**：字典在病历里找到的 HPO 名字，如果落在某个表型的原话里，跟着那个表型的状态；落在所有原话外面的（模型漏掉的），交给模型判断"病历里这个词是在说病人有、没有、亲属有，还是根本不是这个意思"

还有两道安全阀：模型给的原话在病历里找不到的，**不编码**（防止模型编造病历里没有的东西）；模型挑了一个不在候选里的编号，**不采用**。

### 公司里怎么做
"模型负责理解、本体负责编号"就是 grounding；"字典先找、模型再判断"是经典的**规则 + 模型混合**（hybrid）做法：字典便宜、准，但只认字面；模型懂意思，但会编。各取所长。

### 执行
先看字典：不花钱，马上出结果。注意它的问题：分不清"有"和"没有"。

In [ ]:
from raredx.knowledge import hpo as H
from raredx.phenotyping.reader import DictionaryReader
knowledge = H.load(Path("data/raw/hpo/v2026-09-01"))
demo_note = Path("src/raredx/service/notes/PMID_25681079_proband_FKT118.txt").read_text()
print(demo_note)
print("── 字典找到的 ──")
for f in DictionaryReader(knowledge).read(demo_note):
    print(f"  {f.hpo_id}  {f.label:<40} 原话 {f.quote!r}")

字典的问题一眼可见：
- "there are **no** cataracts" 被当成了"有白内障"：字典分不清有和没有
- "floppy"（软）、"raised blood galactose"（血半乳糖高）、"…uridylyltransferase activity was markedly reduced"（酶活性明显降低）这种换了说法的，**一个都没找到**——而后两个正是指向半乳糖血症的关键化验
- "maternal cousin has epilepsy" 它也没找到（"epilepsy" 不是 HPO 登记的说法），这次碰巧没有带偏

下面这一格**花钱（约 ¥0.006）**，可以跳过：用我们的读法现场读同一份病历。

In [ ]:
run("uv run raredx read-note src/raredx/service/notes/PMID_25681079_proband_FKT118.txt 2>&1 | grep -v INFO")

### 结果怎么看
- 每一行：状态（present 有 / absent 没有 / family 亲属）、编号、HPO 名字、病历原话
- "no cataracts" → **absent**，"maternal cousin has epilepsy" → **family**：这两个不会被送去诊断
- "floppy" → Hypotonia，"raised blood galactose" → Hypergalactosemia：字典找不到的，都找到了
- 最后一行是这次花了多少：两次模型调用，不到一分钱（美元）

---

## 第 3 步 · HPO 词条搜索：给模型候选

### 在干什么
把 HPO 里 19,177 个表型词条的名字和同义词（一共 42,938 条）做成一个搜索索引。给一个说法，返回最像的几个词条。

### 背景
第 2 步里"模型只能从候选里挑"，候选从哪来？就从这里来。同义词很重要：HPO 给 "Hearing impairment"（听力障碍）登记了 "Hearing loss"、"Deafness" 等说法，医生写哪一种都能搜到。

### 用了什么技术，为什么
**和第 18 章疾病文字检索同一套**：Qdrant 向量库 + FastEmbed，在本机算，不花钱。
- **语义向量**（bge-small）：意思相近就能找到，"raised blood galactose" 找到 "Hypergalactosemia"（高半乳糖血症），两者没有一个词相同
- **关键词**（BM25）：罕见的专业词按字面准确命中
- 两个排名合起来（RRF）

第 18 章的代码这次改成了"什么记录都能索引"（泛型），一份代码两处用。

只收"说的是同一件事"（EXACT）和"很接近"（RELATED）的同义词，不收"更宽"、"更窄"的——那些会把模型往错误的词条上带。

### 公司里怎么做
这叫**检索增强**用在实体标准化上：先检索候选，再让模型挑（retrieve-then-rerank）。同一套检索基础设施服务多个场景，是平台团队最喜欢的事。

### 执行

In [ ]:
from raredx.knowledge.text_index import FastEmbedder, TextIndex
from raredx.phenotyping.reader import Label
index = TextIndex.load(Path("data/processed/hpo-terms/hpo-2026-09-01"), FastEmbedder(Path("data/models")), Label)
print(f"索引里 {len(index.passages):,} 条名字和同义词\n")
for query in ["floppy infant", "raised blood galactose", "cannot hear well", "fits since birth"]:
    hits = index.search(query, 12)
    seen = list(dict.fromkeys((l.hpo_id, l.name) for l, _ in hits))[:4]
    print(f"{query!r:<26} → " + " | ".join(f"{n} ({i})" for i, n in seen))

### 结果怎么看
- "raised blood galactose" → Hypergalactosemia 排第一；"cannot hear well" → 听力障碍在前三，但**不一定排第一**——所以最后由模型在候选里挑
- **"fits since birth" 搜出来的全不对**："fits" 在英国英语里是"抽搐、发作"，检索不懂这个。这就是为什么第 2 步要先让模型把原话翻成临床术语（"Seizure"），检索先用术语、再用原话。**检索不是万能的，模型和检索各管一段**
- 这一步不花钱，一次搜索 0.1 秒

---

## 第 4 步 · 在开发集上改三版

### 在干什么
先在**开发集**（GeneReviews 10 篇 + GSC-2024 的 22 篇）上试，看错在哪，改，再试。测试集一眼都不看。

### 背景
第一版写出来不可能就是最好的。但改的时候有两个原则：
1. **只在开发集上改**，测试集留到最后考一次（第 20 章讲过为什么）
2. **只改"该改的"错**：有些"错"其实是标注者的习惯，迎合它分数会涨，但读病历并没有变好

### 用了什么技术，为什么
每一版跑完，逐份列出"漏掉的"和"多出来的"，一条条看。下面是三版改了什么：

| 版本 | 改了什么，为什么 | GSC 开发 | GeneReviews |
|---|---|---|---|
| 字典 | 对照 | 52.7% | 78.2% |
| 模型直接写编号 | 对照 | 41.3% | 67.4% |
| v1 | 第一版 | 57.0% | 78.5% |
| v2 | "轻到中度智力障碍"要拆成两端；"肾脏异常"这种笼统的也算；本身就是表型的病名也算（白化病）；原话和 HPO 名字一字不差的直接用 | 61.6% | 81.5% |
| v3 | 加字典核对（第 2 步第 4 点）；字典不用缩写 | 66.4% | 82.8% |
| v3 + 连字符规则 | 字典不从带连字符的词中间找（见下） | 66.7% | 82.2% |

（数字是文档级 micro F1，第 6 步解释。）

**改掉的错，举几个：**
- **"S4"**：GSC 一篇摘要里写脊柱 "L5-S4"，字典把 S4 当成了"第四心音"（S4 是它在 HPO 里登记的缩写）。所以字典不用缩写——缩写交给模型，它看得懂上下文
- **"tonic-clonic seizure"**：在演示病历上看到的。连字符把一个词拆成两半，字典从中间找出了"clonic seizure"（阵挛发作），一个病人没有的表型。改成：名字不能从带连字符的词中间开始或结束。这个是在方案写好之后、考试之前发现的，所以方案里写了补充（第 5 步）
- **"collecting system"**（肾的集合系统）被字典当成了 "Collecting"——HPO 里"囤积癖"的同义词。字典找到、落在原话外面的词，要交给模型判断，模型说"不是这个意思"，就丢掉

**没有去改的"错"：** GSC 的标注习惯是一个短语里所有的 HPO 名字都标。"nevoid basal cell carcinoma syndrome"（痣样基底细胞癌综合征）一个短语，标了"痣"、"癌"、"基底细胞癌"三个。我们只标最具体的那一个。要迎合，就得把每个词条的上位词都加上——GSC 的分数会涨，但 ID-68（真实病历，标注习惯是取最具体的）会掉，而且对诊断没有用。**所以不改，在方案里事先写明：GSC 上会明显低于已发表的方法。**

### 公司里怎么做
这就是**错误分析**（error analysis）驱动的迭代：不是凭感觉改提示词，而是看具体的错，分类，只修有道理的。哪些错是"标注噪音"不去追，叫**不对着测试集调参**（no overfitting to the benchmark）。

### 执行
看一份开发集摘要上 v3 漏掉和多出来的，都是保存下来的结果，不花钱：

In [ ]:
import json
from raredx.phenotyping.evaluation import gold_terms, reader_terms
from raredx.phenotyping.reader import Finding
folders = sorted(Path("data/processed/notes/answers").glob("note-reader-*"), key=lambda p: p.stat().st_mtime)
latest = folders[-1]
doc = next(d for d in corpora.load("gsc-2024-dev", A, T) if d.id == "18177466")
found = [Finding.model_validate(x) for x in json.loads((latest / f"gsc-2024-dev-{doc.id}.json").read_text())]
gold, _, _ = gold_terms(doc, knowledge); ours = reader_terms(found, knowledge)
name = lambda i: knowledge.term(i)[1] if knowledge.current(i) else i
print(doc.text[:500], "…\n")
print("对的  :", ", ".join(sorted(name(i) for i in gold & ours)))
print("漏掉的:", ", ".join(sorted(name(i) for i in gold - ours)))
print("多出的:", ", ".join(sorted(name(i) for i in ours - gold)))

### 结果怎么看
- "漏掉的"里有不少是笼统的上位词（"耳的异常"、"眼的异常"），出自"ophthalmic, orofacial, articular, and auditory manifestations"这种一句话列四个器官的写法——标注者每个都标了一个笼统词条
- 这类错，有的该修（v2 加了"笼统的也算"），有的是标注习惯（不追）。分清楚这两种，是这一步真正的功夫

---

## 第 5 步 · 先写方案，再考

### 在干什么
考测试集之前，把"考什么、怎么判分、和谁比、怎么算赢、我预计多少分"写下来提交：[`docs/phenotyping_plan.md`](../phenotyping_plan.md)。

### 背景
和第 21、23、24 章一样：先考完再决定怎么算，总能找到一种对自己有利的算法。先写下来，就没有这个余地。

### 用了什么技术，为什么
方案的要点：

| | |
|---|---|
| 主要问题 | ID-68 上，我们的读法的 micro F1，是否**同时**高于字典和模型直接写编号 |
| 判分 | 按文档比词条集合（和 PhenoTagger、PhenoBERT、AutoPCR 一样）；只算表型异常之下的词条；标注的旧编号先换成新版本的 |
| 检验 | 逐份病历比 F1，谁高算谁赢，符号检验，p < 0.05 |
| 和已发表的比 | AutoPCR 文档级 F1：ID-68 80.3，GSC-2024 83.5；只引用，不重跑，不做检验 |
| 我的预期 | ID-68：我们 75–82%，字典 60–70%，模型直接写 40–60%；GSC：我们 63–70%，明显低于 AutoPCR |

**micro 和 macro**：micro 是把所有病历的"对、错、漏"加在一起再算；macro 是每份病历先算一个分，再平均。病历长短差很多时，两者会不一样。两种都报，主要看 micro（AutoPCR 报的就是这个）。

**F1**：精确率（给出的里面有多少是对的）和召回率（该找的找到了多少）的调和平均。只追求一个很容易：什么都不给，精确率就没法错；什么都给，召回率就满了。F1 逼着两个都要好。

**考前的补充**：连字符的问题是方案提交以后、考试之前发现的。改了读法，就在方案里加了一段带日期的补充，说明改了什么、为什么、开发集重跑的结果。**改是可以改的，但要在考之前，而且要留下记录。**

### 公司里怎么做
预注册在 A/B 实验里叫**实验设计评审**：上线实验之前写好指标、样本量、判定规则。事后改规则要单独说明，否则结论不被认可。

---

## 第 6 步 · 考试结果

### 在干什么
三种读法各读一遍两套测试题（`--final`，每种一次），按方案判分。

### 背景
方案在考之前就写好了：主要问题是"在真实病历（ID-68）上，我们的读法是否**同时**胜过字典和模型直接写编号"。

### 执行
六次考试都记在 MLflow 里。下面这一格从记录里读出分数，再逐份病历配对比较（不花钱）：

In [ ]:
from mlflow import MlflowClient
from raredx.evals.scoring import p_value
from raredx.phenotyping.evaluation import paired, per_document
uri = "sqlite:///mlflow.db"
client = MlflowClient(uri)
runs = {}
for corpus in ("id-68", "gsc-2024-test"):
    experiment = client.get_experiment_by_name(f"eval/notes-{corpus}")
    for r in client.search_runs([experiment.experiment_id], order_by=["start_time DESC"]):
        reader = r.data.params["reader"]
        if r.data.params.get("corpus") == corpus and (corpus, reader) not in runs:
            runs[corpus, reader] = r
for corpus in ("id-68", "gsc-2024-test"):
    print(f"\n── {corpus} ──")
    for reader in ("note-reader", "dictionary", "model-direct"):
        m = runs[corpus, reader].data.metrics
        print(f"  {reader:<13} 精确率 {m['micro/precision']:6.1%}  召回率 {m['micro/recall']:6.1%}  F1 {m['micro/f1']:6.1%}")
    ours = per_document(uri, runs[corpus, "note-reader"].info.run_id)
    for other in ("dictionary", "model-direct"):
        wins, losses, p = paired(ours, per_document(uri, runs[corpus, other].info.run_id))
        print(f"  逐份比 F1：我们 vs {other:<13} {wins} : {losses}  {p_value(p)}")

### 结果怎么看

**主要结论：不成立。**

| ID-68（真实病历） | 精确率 | 召回率 | F1 | 逐份比 F1 |
|---|---|---|---|---|
| 只查字典 | 89.9% | 60.5% | **72.3%** | 我们 29 : 字典 38，p = 0.33 |
| **我们的读法** | 64.7% | 80.3% | **71.7%** | |
| 模型直接写编号 | 50.7% | 60.8% | 55.3% | 我们 58 : 5，p < 0.001 |

| GSC-2024（论文摘要） | F1 | 逐份比 F1 |
|---|---|---|
| **我们的读法** | **71.5%** | |
| 只查字典 | 58.6% | 我们 122 : 31，p < 0.001 |
| 模型直接写编号 | 47.3% | 我们 160 : 11，p < 0.001 |

- **在真实病历上，我们没有赢过字典**：F1 打平，逐份比字典还多赢几份（不显著）。我预计我们 75%–82%、字典 60%–70%，两个都猜错了
- **模型直接写编号在两套题上都最差**：这证实了"不让模型自己写编号"是对的
- **论文摘要上明显赢了两个对照**，但比已发表的 AutoPCR（83.5%）低 12 个百分点——事先写明了会低
- 和 AutoPCR 在 ID-68 上（80.3%）比，低 9 个百分点

**为什么在真实病历上没赢？** 考完以后看了我们"多给"的 347 个词条（这是考后分析，不改结论）：

| 类别 | 数量 | 例子 |
|---|---|---|
| 病历里写了、标注者没有标的话 | 185（53%） | "floppy" → 肌张力低下；"IQ was estimated to be 20-30" → 重度智力障碍 |
| 同一个表型、上下级不同 | 101（29%） | "thinning of the corpus callosum" → 新版 HPO 的"胼胝体变薄"，标注用的是 2019 年版 |
| 同一段话、不同的词条 | 53（15%） | "no tears when she cries" → 我们"无泪"，标注"流泪"（标注错了）；也有我们选错的 |
| 亲属的、否认的 | 8 | |

还记得第 1 步里那份病历吗？"floppy"没有被标。这样的情况，我们的读法每找对一次，就被算"错"一次。

如果"直接上一级、下一级也算对"（PhenoBERT 团队标 ID-68 时就这样标），三种读法同样算：我们 79.6%、字典 78.9%、模型直接写编号 66.7%。**还是打平。**

所以结论是：**两种做法取舍相反**。
- 字典：给的几乎都对，但漏掉三分之一；分不清有和没有——病历里否认的 7 个症状，它把 3 个当成了"有"
- 我们的读法：找得全，分得清有、没有、亲属有；多给的词条里一半是标注漏标的话

对诊断来说哪种代价更大？这次没量：要拿读出来的词条去诊断、和确诊比，得有带确诊结果的公开病历，ID-68 没有。

### 公司里怎么做
**负面结果照实报告。** 方案事先写了"怎么算赢"，结果没赢，就写"没赢"，再另外写考后分析帮助理解——但不能用考后分析把结论翻过来，更不能考完再改代码重考。评审最看重的就是这一点：**你的数字经不经得起追问**。

两个系统 F1 打平、取舍相反时，选哪个要看场景：漏掉症状代价大（诊断），就要召回；误报代价大（自动编码计费），就要精确。也可以两个都用：字典的结果当"高把握"，我们的读法补充，再让人确认——演示页面就是这么做的（每个词条都给人勾选）。

### 面试怎么说
> "On ID-68, real clinical notes, my reader tied the dictionary on F1 — 71.7 against 72.3 — so the pre-registered primary claim failed, and I report it that way. It beat the model writing IDs itself by 58 notes to 5, and on the GSC-2024 abstracts it beat both baselines clearly. Looking afterwards, half of the terms counted against it are findings the annotators didn't annotate — 'floppy', an IQ of 20–30 — and the dictionary can't tell a denied finding from a present one. So the two trade recall for precision; which matters more would need notes with confirmed diagnoses, which ID-68 doesn't have."

---

## 第 7 步 · 做进服务，病历不留底

### 在干什么
在第 22 章的服务上加一个接口：`POST /v1/terms/from-note`，收一段病历，返回读出来的词条，给人核对。

### 背景
第 22 章做服务的时候，故意**只收编号，不收任何自由文字**：名字、生日、病历都没地方填，也就不可能误传、误存。现在要收病历了，这个保证得用别的办法重新做到：

| 风险 | 怎么防 |
|---|---|
| 病历写进日志 | 日志只记"多少个字、找到几个词条、花了多少"，不记一个字 |
| 病历写进追踪记录（MLflow 会记下每一步的输入和输出，包括发给模型的整段病历） | 读病历的时候，每一步结束、写进记录**之前**，把输入输出换成"不记录" |
| 出错信息里带出病历（模型的回答引用了病历原话，格式错误时会出现在报错里） | 报错只说"格式不对"，不带回答内容 |
| 病历发给模型服务商 | 这是读病历必须的，挡不住。所以这个接口**默认关闭**，部署时明确打开；页面和文档写明"不要贴真实病人的病历"，真用要和服务商签数据协议，或者用自己部署的模型 |

诊断接口 `/v1/diagnose` 不变：**还是只收编号**。病历读成词条、人核对过以后，送去诊断的只有编号。

### 用了什么技术，为什么
**MLflow 的 span processor**：MLflow 提供的钩子，每一步（span）结束、写进记录之前调用一次，专门用来遮掉敏感数据。我们的规则很简单：正在读病历的时候结束的每一步，输入输出都换成一句"不记录"。记录里还留着：走了哪几步、每步多久、每次模型调用用了多少 token、最后找到了哪些编号。

"正在读病历"怎么知道？用 Python 的 **context variable**（上下文变量）：读之前设上，读完撤掉。每一步结束时在同一个线程里，看得到这个标记。

**测试怎么证明**：测试里用真正的 OpenAI 客户端，只是把网络换成假的，这样 MLflow 会像线上一样自动记录每次模型调用。然后把整条记录取出来，**检查病历里的名字、生日、症状原话一个都不在里面**。演示服务真实跑一次以后，也这样查了一遍。

### 公司里怎么做
医疗数据里的病人信息叫 **PHI**（受保护的健康信息）。原则是**数据最小化**：只收需要的，只留必须留的。日志和追踪里遮掉敏感字段（PII redaction）是可观测性平台的标准功能；"默认关闭、显式打开"叫 **secure by default**。

### 执行
跑这部分的测试（不花钱）：

In [ ]:
run("uv run pytest tests/service/test_service_notes.py -rA 2>&1 | grep -E '^PASSED|^FAILED| passed| failed'")

### 结果怎么看
- `test_the_trace_keeps_the_steps_and_the_codes_and_no_word_of_the_note`：最重要的一条。记录里有每一步、有编号，没有病历的一个字
- `test_a_reply_the_model_cannot_put_right_fails_without_quoting_it`：模型回答格式错误时，报错和记录里也没有病历
- `test_a_service_that_takes_no_notes_has_no_such_endpoint`：默认关闭时，这个接口根本不存在

---

## 第 8 步 · 演示：贴一段病历 → 出诊断

### 在干什么
演示页面加一个入口"Start from a clinical note"：贴病历 → 读 → 在病历上标出每个表型的原话 → 勾选 → 诊断。

### 背景
这是最终录屏要展示的完整流程，也是这个项目在临床上用起来的样子。

演示用的两份病历是**为演示写的**：照着两个已发表病人的表型，用医生的口吻写成（半乳糖血症、Holt-Oram 综合征），页面上写明"不是真实病历"。半乳糖血症那个病人是第 22 章的"agent 救回来的那道题"：只靠表型排名找不到，agent 排第一。

### 用了什么技术，为什么
- 病历原话在页面上**按状态上色**：绿色 = 病人有，灰色划线 = 病历否认，蓝色 = 亲属有。医生一眼能看出读得对不对
- 下面一张表，每个词条一个勾：病人有的默认勾上，否认的、亲属的不勾。**医生可以改**——人在回路里（human in the loop）
- 显示病历时全部转义：贴进来什么就显示什么，不会被当成网页代码执行

### 公司里怎么做
AI 读出来的东西给人核对、人确认后再用，叫 **human-in-the-loop**。在医疗场景里这是硬要求：系统给建议，决定是医生做的。

### 执行
真实跑过一次（演示服务，2026-10-14）：

| 步骤 | 结果 | 用时 | 花费 |
|---|---|---|---|
| 读病历 | 13 个表型：9 个"有"、1 个"没有"（白内障）、1 个"亲属有"（癫痫）、2 个没编码 | 4.4 秒 | $0.0008 |
| 诊断（9 个编号，病人自己的论文排除在外） | **第 1 名：半乳糖血症**（就是论文确诊的） | 13 秒 | $0.005 |

自己试：开两个终端，

```
RAREDX_SERVICE_READ_NOTES=true uv run raredx serve
uv run raredx demo
```

浏览器打开 http://127.0.0.1:8501 ，"Patient" 选 **Start from a clinical note**。或者 `docker compose up --build`，演示部署已经打开了读病历。

---

## 这一章做完，项目多了什么

| | |
|---|---|
| 新能力 | 病历文字 → HPO 词条 → 诊断，完整流程 |
| 新代码 | `src/raredx/phenotyping/`（读法、语料、评测）；服务接口 `/v1/terms/from-note`；`service/privacy.py` |
| 新命令 | `raredx data hpo-index`、`raredx read-note`、`raredx eval notes` |
| 测试 | 新增约 50 个；病历不进记录有专门的测试 |
| 花费 | 开发约 $0.15，正式考试 $0.37 |

## 一句话

**大模型负责读懂，本体负责编号，字典负责兜底，人负责确认。**

## 面试题

1. **How do you turn free-text clinical notes into ontology codes without the LLM hallucinating IDs?**
   > "The model never writes an ID. It extracts each finding with its exact words and status; exact name matches are coded directly; for the rest, a hybrid search over HPO's names and synonyms gives candidates and the model picks one or none. A finding whose words aren't in the note isn't coded. A dictionary pass catches names the model skipped, and the model checks those in context."

2. **How do you handle negation and family history?**
   > "Each finding carries a status — present, absent, family — from the model, which reads context far better than NegEx-style rules; dictionary hits inside a finding's words inherit its status. Only present findings are pre-selected for diagnosis, and a clinician confirms them in the UI."

3. **How did you evaluate it?**
   > "On public hand-annotated corpora — ID-68, real clinical notes, and GSC-2024 — at the document level, the same measure published methods report, with a pre-registered plan, a dictionary and a model-writes-IDs baseline, and a paired test per note."

4. **The service now takes clinical notes. What about privacy?**
   > "Off by default. Logs carry counts only. An MLflow span processor redacts every span's inputs and outputs while a note is read, so traces keep timings, token counts and the codes, not the text — a test drives the real OpenAI SDK with autologging and asserts no word of the note is stored. The note does go to the model provider, so production needs a BAA-covered or self-hosted model."

**下一章 27 · 交付**：结果报告页、架构图、录屏。